# Spatial Resolution Analysis - Pixel Size Calculation

This notebook calculates the actual physical size of hyperspectral image pixels by analyzing the georeferenced positions.

**Method:**
- Extract georeferenced positions (NED coordinates) for each pixel
- Calculate distances between adjacent pixels
- Compute across-track pixel size (distance between pixels in same row)
- Compute along-track pixel size (distance between pixels in adjacent rows)
- Analyze how pixel size varies with altitude and position

In [ ]:
import importlib
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

# Add mjosa_code root to path
mjosa_code_root = Path.cwd().parent  # Up to mjosa_code root
sys.path.insert(0, str(mjosa_code_root))

# Import from mjosa_code (NEW structure)
from utils.uhi import georef
from utils.common import config

importlib.reload(georef)
from utils.uhi.georef import *

## Load Transect Data

In [ ]:
# Load transect 057 (default)
transect = load_transect(config.TRANSECT_057_OUTPUT)
transect.list_files()

# Select file
cube = transect.select_files(["rad_uhi_20241029_115057_5"])
cube.describe()

## Define Pixel Resolution Analysis Function

In [ ]:
def calculate_pixel_resolution(cube, track_start, track_end, apply_alignment=True):
    """
    Calculate pixel spatial resolution from georeferenced positions.

    Parameters
    ----------
    cube : CombinedTransectCube
        Loaded hyperspectral cube
    track_start : int
        Start track index
    track_end : int
        End track index (exclusive)
    apply_alignment : bool
        Apply UHI alignment shift

    Returns
    -------
    dict
        Dictionary with pixel resolution statistics
    """
    print(f"\n{'='*80}")
    print(f"PIXEL SPATIAL RESOLUTION ANALYSIS")
    print(f"{'='*80}")
    print(
        f"Track range: {track_start} to {track_end} ({track_end - track_start} tracks)"
    )
    print(f"Alignment shift: {apply_alignment}")

    # Get georeferenced positions for all pixels in NED coordinates
    # This extracts the actual coordinates that plot_georef uses
    positions_ned = []

    for file_idx, file_obj in enumerate(cube.files):
        # Get position data from file
        nav_data = file_obj.navigation

        # Get track range for this file
        file_track_start = max(track_start, file_obj.track_start_index)
        file_track_end = min(track_end, file_obj.track_start_index + file_obj.n_lines)

        if file_track_end <= file_track_start:
            continue

        # Extract positions for each pixel
        local_start = file_track_start - file_obj.track_start_index
        local_end = file_track_end - file_obj.track_start_index

        # Get camera poses and pixel directions
        camera_pos = nav_data["cam_pos_ecef"][local_start:local_end]  # (n_tracks, 3)
        pixel_dirs = file_obj.ray_direction_ecef[
            local_start:local_end
        ]  # (n_tracks, n_pixels, 3)
        ranges = file_obj.ranges[local_start:local_end]  # (n_tracks, n_pixels)

        # Calculate pixel positions in ECEF
        # position = camera_pos + range * direction
        n_tracks, n_pixels, _ = pixel_dirs.shape
        pixel_pos_ecef = (
            camera_pos[:, np.newaxis, :] + ranges[:, :, np.newaxis] * pixel_dirs
        )

        # Convert ECEF to NED using pyproj
        from pyproj import Transformer

        transformer_ecef_to_latlon = Transformer.from_crs(
            "EPSG:4978", "EPSG:4326", always_xy=True  # ECEF  # WGS84
        )

        # Flatten for transformation
        flat_ecef = pixel_pos_ecef.reshape(-1, 3)
        lons, lats, alts = transformer_ecef_to_latlon.transform(
            flat_ecef[:, 0], flat_ecef[:, 1], flat_ecef[:, 2]
        )

        # Convert to NED (local East-North-Up coordinates)
        lat0, lon0 = config.MJOSA_ORIGIN
        transformer_latlon_to_utm = Transformer.from_crs(
            "EPSG:4326", "EPSG:32632", always_xy=True  # WGS84  # UTM 32N
        )

        east, north, _ = transformer_latlon_to_utm.transform(lons, lats, alts)
        east0, north0, _ = transformer_latlon_to_utm.transform([lon0], [lat0], [0])

        # Convert to local NED (North-East-Down)
        local_north = north - north0[0]
        local_east = east - east0[0]
        local_down = -alts  # Down is negative altitude

        # Apply UHI alignment if requested
        if apply_alignment:
            local_east += config.UHI_ALIGNMENT_DX
            local_north += config.UHI_ALIGNMENT_DY

        # Reshape back to (n_tracks, n_pixels, 3)
        ned_positions = np.stack([local_north, local_east, local_down], axis=1)
        ned_positions = ned_positions.reshape(n_tracks, n_pixels, 3)

        positions_ned.append(ned_positions)

    # Concatenate all files
    all_positions = np.concatenate(positions_ned, axis=0)  # (total_tracks, n_pixels, 3)
    n_tracks, n_pixels, _ = all_positions.shape

    print(f"\nExtracted positions for {n_tracks} tracks × {n_pixels} pixels")

    # Calculate across-track pixel spacing (distance between adjacent pixels in same track)
    across_track_distances = []
    for track_idx in range(n_tracks):
        for pixel_idx in range(n_pixels - 1):
            pos1 = all_positions[track_idx, pixel_idx, :2]  # North, East
            pos2 = all_positions[track_idx, pixel_idx + 1, :2]
            dist = np.linalg.norm(pos2 - pos1)
            if dist > 0 and dist < 1.0:  # Filter outliers (> 1m seems unreasonable)
                across_track_distances.append(dist)

    across_track_distances = np.array(across_track_distances)

    # Calculate along-track pixel spacing (distance between same pixel in adjacent tracks)
    along_track_distances = []
    for track_idx in range(n_tracks - 1):
        for pixel_idx in range(n_pixels):
            pos1 = all_positions[track_idx, pixel_idx, :2]
            pos2 = all_positions[track_idx + 1, pixel_idx, :2]
            dist = np.linalg.norm(pos2 - pos1)
            if dist > 0 and dist < 2.0:  # Filter outliers (> 2m seems unreasonable)
                along_track_distances.append(dist)

    along_track_distances = np.array(along_track_distances)

    # Calculate altitude statistics
    altitudes = -all_positions[:, :, 2].flatten()  # Convert down to altitude
    altitudes = altitudes[altitudes > 0]  # Filter valid altitudes

    # Compute statistics
    results = {
        "across_track": {
            "mean": np.mean(across_track_distances),
            "median": np.median(across_track_distances),
            "std": np.std(across_track_distances),
            "min": np.min(across_track_distances),
            "max": np.max(across_track_distances),
            "data": across_track_distances,
        },
        "along_track": {
            "mean": np.mean(along_track_distances),
            "median": np.median(along_track_distances),
            "std": np.std(along_track_distances),
            "min": np.min(along_track_distances),
            "max": np.max(along_track_distances),
            "data": along_track_distances,
        },
        "altitude": {
            "mean": np.mean(altitudes),
            "median": np.median(altitudes),
            "std": np.std(altitudes),
            "min": np.min(altitudes),
            "max": np.max(altitudes),
            "data": altitudes,
        },
        "positions": all_positions,
    }

    # Print summary
    print(f"\n{'='*80}")
    print(f"PIXEL RESOLUTION STATISTICS (in meters)")
    print(f"{'='*80}")

    print(f"\n📏 ACROSS-TRACK PIXEL SIZE (width):")
    print(f"   Mean:   {results['across_track']['mean']*100:.2f} cm")
    print(f"   Median: {results['across_track']['median']*100:.2f} cm")
    print(f"   Std:    {results['across_track']['std']*100:.2f} cm")
    print(f"   Min:    {results['across_track']['min']*100:.2f} cm")
    print(f"   Max:    {results['across_track']['max']*100:.2f} cm")

    print(f"\n📏 ALONG-TRACK PIXEL SIZE (height):")
    print(f"   Mean:   {results['along_track']['mean']*100:.2f} cm")
    print(f"   Median: {results['along_track']['median']*100:.2f} cm")
    print(f"   Std:    {results['along_track']['std']*100:.2f} cm")
    print(f"   Min:    {results['along_track']['min']*100:.2f} cm")
    print(f"   Max:    {results['along_track']['max']*100:.2f} cm")

    print(f"\n📐 PIXEL ASPECT RATIO:")
    aspect_ratio = results["along_track"]["mean"] / results["across_track"]["mean"]
    print(f"   Along-track / Across-track: {aspect_ratio:.2f}")

    print(f"\n🗻 ALTITUDE ABOVE SEAFLOOR:")
    print(f"   Mean:   {results['altitude']['mean']:.2f} m")
    print(f"   Median: {results['altitude']['median']:.2f} m")
    print(f"   Std:    {results['altitude']['std']:.2f} m")
    print(f"   Min:    {results['altitude']['min']:.2f} m")
    print(f"   Max:    {results['altitude']['max']:.2f} m")

    print(f"\n💡 MEAN PIXEL FOOTPRINT:")
    print(
        f"   {results['across_track']['mean']*100:.2f} cm × {results['along_track']['mean']*100:.2f} cm"
    )
    area_cm2 = (results["across_track"]["mean"] * 100) * (
        results["along_track"]["mean"] * 100
    )
    print(f"   Area: {area_cm2:.2f} cm²")

    print(f"\n{'='*80}")

    return results

## Calculate Pixel Resolution for Transect 057

In [ ]:
# Calculate pixel resolution
resolution_results = calculate_pixel_resolution(
    cube=cube,
    track_start=config.UHI_TRACK_RANGE[0],
    track_end=config.UHI_TRACK_RANGE[1],
    apply_alignment=True,
)

## Visualize Pixel Size Distributions

In [ ]:
# Create comprehensive visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Across-track pixel size histogram
ax = axes[0, 0]
ax.hist(
    resolution_results["across_track"]["data"] * 100,
    bins=50,
    color="steelblue",
    alpha=0.7,
    edgecolor="black",
)
ax.axvline(
    resolution_results["across_track"]["mean"] * 100,
    color="red",
    linestyle="--",
    linewidth=2,
    label=f"Mean: {resolution_results['across_track']['mean']*100:.2f} cm",
)
ax.axvline(
    resolution_results["across_track"]["median"] * 100,
    color="orange",
    linestyle="--",
    linewidth=2,
    label=f"Median: {resolution_results['across_track']['median']*100:.2f} cm",
)
ax.set_xlabel("Pixel Size (cm)", fontsize=12, fontweight="bold")
ax.set_ylabel("Frequency", fontsize=12, fontweight="bold")
ax.set_title("Across-Track Pixel Size Distribution", fontsize=14, fontweight="bold")
ax.legend()
ax.grid(alpha=0.3)

# 2. Along-track pixel size histogram
ax = axes[0, 1]
ax.hist(
    resolution_results["along_track"]["data"] * 100,
    bins=50,
    color="forestgreen",
    alpha=0.7,
    edgecolor="black",
)
ax.axvline(
    resolution_results["along_track"]["mean"] * 100,
    color="red",
    linestyle="--",
    linewidth=2,
    label=f"Mean: {resolution_results['along_track']['mean']*100:.2f} cm",
)
ax.axvline(
    resolution_results["along_track"]["median"] * 100,
    color="orange",
    linestyle="--",
    linewidth=2,
    label=f"Median: {resolution_results['along_track']['median']*100:.2f} cm",
)
ax.set_xlabel("Pixel Size (cm)", fontsize=12, fontweight="bold")
ax.set_ylabel("Frequency", fontsize=12, fontweight="bold")
ax.set_title("Along-Track Pixel Size Distribution", fontsize=14, fontweight="bold")
ax.legend()
ax.grid(alpha=0.3)

# 3. Altitude distribution
ax = axes[1, 0]
ax.hist(
    resolution_results["altitude"]["data"],
    bins=50,
    color="coral",
    alpha=0.7,
    edgecolor="black",
)
ax.axvline(
    resolution_results["altitude"]["mean"],
    color="red",
    linestyle="--",
    linewidth=2,
    label=f"Mean: {resolution_results['altitude']['mean']:.2f} m",
)
ax.axvline(
    resolution_results["altitude"]["median"],
    color="orange",
    linestyle="--",
    linewidth=2,
    label=f"Median: {resolution_results['altitude']['median']:.2f} m",
)
ax.set_xlabel("Altitude Above Seafloor (m)", fontsize=12, fontweight="bold")
ax.set_ylabel("Frequency", fontsize=12, fontweight="bold")
ax.set_title("Altitude Distribution", fontsize=14, fontweight="bold")
ax.legend()
ax.grid(alpha=0.3)

# 4. Comparison: across vs along track
ax = axes[1, 1]
categories = ["Across-Track\n(Width)", "Along-Track\n(Height)"]
means = [
    resolution_results["across_track"]["mean"] * 100,
    resolution_results["along_track"]["mean"] * 100,
]
stds = [
    resolution_results["across_track"]["std"] * 100,
    resolution_results["along_track"]["std"] * 100,
]

bars = ax.bar(
    categories,
    means,
    yerr=stds,
    color=["steelblue", "forestgreen"],
    alpha=0.7,
    edgecolor="black",
    linewidth=2,
    capsize=10,
)
ax.set_ylabel("Mean Pixel Size (cm)", fontsize=12, fontweight="bold")
ax.set_title("Pixel Size Comparison", fontsize=14, fontweight="bold")
ax.grid(axis="y", alpha=0.3)

# Add value labels on bars
for bar, mean, std in zip(bars, means, stds):
    height = bar.get_height()
    ax.text(
        bar.get_x() + bar.get_width() / 2.0,
        height + std + 0.5,
        f"{mean:.2f} ± {std:.2f} cm",
        ha="center",
        va="bottom",
        fontweight="bold",
        fontsize=10,
    )

aspect_ratio = (
    resolution_results["along_track"]["mean"]
    / resolution_results["across_track"]["mean"]
)
ax.text(
    0.5,
    0.95,
    f"Aspect Ratio: {aspect_ratio:.2f}:1",
    transform=ax.transAxes,
    ha="center",
    va="top",
    bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.5),
    fontsize=11,
    fontweight="bold",
)

plt.tight_layout()
plt.show()

print(f"\n✅ Plots generated successfully!")

## Summary Statistics Table

In [ ]:
# Print comprehensive summary table
print(f"\n{'='*80}")
print(f"COMPREHENSIVE PIXEL RESOLUTION SUMMARY")
print(f"{'='*80}")
print(f"\n{'Metric':<30} {'Across-Track':<20} {'Along-Track':<20}")
print(f"{'-'*70}")
print(
    f"{'Mean (cm)':<30} {resolution_results['across_track']['mean']*100:>19.2f} {resolution_results['along_track']['mean']*100:>19.2f}"
)
print(
    f"{'Median (cm)':<30} {resolution_results['across_track']['median']*100:>19.2f} {resolution_results['along_track']['median']*100:>19.2f}"
)
print(
    f"{'Std Dev (cm)':<30} {resolution_results['across_track']['std']*100:>19.2f} {resolution_results['along_track']['std']*100:>19.2f}"
)
print(
    f"{'Min (cm)':<30} {resolution_results['across_track']['min']*100:>19.2f} {resolution_results['along_track']['min']*100:>19.2f}"
)
print(
    f"{'Max (cm)':<30} {resolution_results['across_track']['max']*100:>19.2f} {resolution_results['along_track']['max']*100:>19.2f}"
)
print(f"{'='*80}")

print(f"\n📊 KEY FINDINGS:")
print(
    f"   • Typical pixel footprint: {resolution_results['across_track']['mean']*100:.2f} cm × {resolution_results['along_track']['mean']*100:.2f} cm"
)
print(
    f"   • Pixel aspect ratio: {resolution_results['along_track']['mean'] / resolution_results['across_track']['mean']:.2f}:1 (height:width)"
)
print(
    f"   • Mean altitude: {resolution_results['altitude']['mean']:.2f} m above seafloor"
)
print(
    f"   • Pixel area: {(resolution_results['across_track']['mean']*100) * (resolution_results['along_track']['mean']*100):.2f} cm²"
)
print(f"{'='*80}")